## T01

<!-- merged from: T01.ipynb (Jupy Tools) -->

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
n = 3
# 3 query qubits + 1 ancilla qubit, 3 classical bits
qc_const = QuantumCircuit(n + 1, n)
# Initialize input in |+>^3 and ancilla in |->
qc_const.x(n)
qc_const.h(range(n + 1))
qc_const.barrier()
# Constant Oracle f(x) = 0: Target register unchanged (identity / no gates)
qc_const.barrier()
# Apply Hadamard to query qubits and measure
qc_const.h(range(n))
qc_const.measure(range(n), range(n))
sim = AerSimulator()
counts_const = sim.run(qc_const, shots=1000).result().get_counts()
print("--- 3-Qubit Deutsch-Jozsa Circuit (Constant-0) ---")
print(qc_const)
print("\nMeasurement Readout:", counts_const)
is_constant = "000" in counts_const and len(counts_const) == 1
print("Classification:", "Constant Function (f is constant)" if is_constant else "Balanced Function")

--- 3-Qubit Deutsch-Jozsa Circuit (Constant-0) ---
     ┌───┐      ░  ░ ┌───┐┌─┐      
q_0: ┤ H ├──────░──░─┤ H ├┤M├──────
     ├───┤      ░  ░ ├───┤└╥┘┌─┐   
q_1: ┤ H ├──────░──░─┤ H ├─╫─┤M├───
     ├───┤      ░  ░ ├───┤ ║ └╥┘┌─┐
q_2: ┤ H ├──────░──░─┤ H ├─╫──╫─┤M├
     ├───┤┌───┐ ░  ░ └───┘ ║  ║ └╥┘
q_3: ┤ X ├┤ H ├─░──░───────╫──╫──╫─
     └───┘└───┘ ░  ░       ║  ║  ║ 
c: 3/══════════════════════╩══╩══╩═
                           0  1  2 

Measurement Readout: {'000': 1000}
Classification: Constant Function (f is constant)


## T02

<!-- merged from: T02.ipynb (Jupy Tools) -->

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
n = 3
qc_balanced = QuantumCircuit(n + 1, n)
# State initialization: |+>^3 and |->
qc_balanced.x(n)
qc_balanced.h(range(n + 1))
qc_balanced.barrier()
# Balanced Oracle: Parity function f(x) = x0 ^ x1 ^ x2
for i in range(n):
 qc_balanced.cx(i, n)
qc_balanced.barrier()
# Recombination & readout
qc_balanced.h(range(n))
qc_balanced.measure(range(n), range(n))
counts_bal = sim.run(qc_balanced, shots=1000).result().get_counts()
print("--- 3-Qubit Deutsch-Jozsa Circuit (Balanced Parity) ---")
print(qc_balanced)
print("\nMeasurement Readout:", counts_bal)
measured_key = list(counts_bal.keys())[0]
print("Classification:", "Balanced Function (Non-zero state detected)" if measured_key != "000" else "Constant Function (Zero state detected)") 

--- 3-Qubit Deutsch-Jozsa Circuit (Balanced Parity) ---
     ┌───┐      ░                 ░ ┌───┐┌─┐      
q_0: ┤ H ├──────░───■─────────────░─┤ H ├┤M├──────
     ├───┤      ░   │             ░ ├───┤└╥┘┌─┐   
q_1: ┤ H ├──────░───┼────■────────░─┤ H ├─╫─┤M├───
     ├───┤      ░   │    │        ░ ├───┤ ║ └╥┘┌─┐
q_2: ┤ H ├──────░───┼────┼────■───░─┤ H ├─╫──╫─┤M├
     ├───┤┌───┐ ░ ┌─┴─┐┌─┴─┐┌─┴─┐ ░ └───┘ ║  ║ └╥┘
q_3: ┤ X ├┤ H ├─░─┤ X ├┤ X ├┤ X ├─░───────╫──╫──╫─
     └───┘└───┘ ░ └───┘└───┘└───┘ ░       ║  ║  ║ 
c: 3/═════════════════════════════════════╩══╩══╩═
                                          0  1  2 

Measurement Readout: {'111': 1000}
Classification: Balanced Function (Non-zero state detected)


## T03 (1)

<!-- merged from: T03 (1).ipynb (Jupy Tools) -->

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
def build_dj_oracle(n_qubits, oracle_type="constant"):
 oracle_qc = QuantumCircuit(n_qubits + 1, name=f"DJ_{oracle_type}_{n_qubits}")
 if oracle_type == "constant":
 # f(x) = 1 (X gate on ancilla) or f(x) = 0 (identity)
  oracle_qc.x(n_qubits)
 elif oracle_type == "balanced":
 # Balanced: CX from all input qubits to ancilla
  for q in range(n_qubits):
   oracle_qc.cx(q, n_qubits)
 return oracle_qc
def run_dj_experiment(n_qubits, oracle_type):
 qc = QuantumCircuit(n_qubits + 1, n_qubits)
 qc.x(n_qubits)
 qc.h(range(n_qubits + 1))
 qc.compose(build_dj_oracle(n_qubits, oracle_type), inplace=True)
 qc.h(range(n_qubits))
 qc.measure(range(n_qubits), range(n_qubits))
 res = AerSimulator().run(qc, shots=500).result().get_counts()
 readout = list(res.keys())[0]
 inferred = "Constant" if readout == "0" * n_qubits else "Balanced"
 return readout, inferred
print("--- Generalized Deutsch-Jozsa Multi-Width Evaluation ---")
print(f"{'n (Qubits)':<12}{'Oracle Type':<16}{'Measured Output':<20}{'Verdict':<12}{'Status'}")
print("=" * 68)
for n_val in range(2, 6):
 for o_type in ["constant", "balanced"]:
  out_str, verd = run_dj_experiment(n_val, o_type)
 status = "PASS" if verd.lower() == o_type else "FAIL"
 print(f"{n_val:<12}{o_type.capitalize():<16}{out_str:<20}{verd:<12}{status}")

--- Generalized Deutsch-Jozsa Multi-Width Evaluation ---
n (Qubits)  Oracle Type     Measured Output     Verdict     Status
2           Balanced        11                  Balanced    PASS
3           Balanced        111                 Balanced    PASS
4           Balanced        1111                Balanced    PASS
5           Balanced        11111               Balanced    PASS


## T04 (1)

<!-- merged from: T04 (1).ipynb (Jupy Tools) -->

In [ ]:
import numpy as np
# Comparative scaling analysis: Classical worst-case vs. Quantum
n_range = np.arange(2, 11)
classical_worst = [2**(n - 1) + 1 for n in n_range]
quantum_queries = [1 for _ in n_range]
print("--- Query Complexity Comparison (Deutsch-Jozsa) ---")
print(f"{'n (Bits)':<10}{'Domain (2^n)':<18}{'Classical Worst-Case [2^(n-1)+1]':<35}{'Quantum DJ'}")
print("=" * 75)
for i, n_val in enumerate(n_range):
 print(f"{n_val:<10}{2**n_val:<18}{classical_worst[i]:<35}{quantum_queries[i]}")
print("=" * 75)
print(f"Exponential Speedup Factor at n=10: {classical_worst[-1] / quantum_queries[-1]}x")

--- Query Complexity Comparison (Deutsch-Jozsa) ---
n (Bits)  Domain (2^n)      Classical Worst-Case [2^(n-1)+1]   Quantum DJ
2         4                 3                                  1
3         8                 5                                  1
4         16                9                                  1
5         32                17                                 1
6         64                33                                 1
7         128               65                                 1
8         256               129                                1
9         512               257                                1
10        1024              513                                1
Exponential Speedup Factor at n=10: 513.0x


## T05 (1)

<!-- merged from: T05 (1).ipynb (Jupy Tools) -->

In [ ]:
import random
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
def build_random_balanced_oracle(n_qubits):
 oracle_qc = QuantumCircuit(n_qubits + 1, name="RandomBalanced")
 # Pick a random non-zero bitmask b in [1, 2^n - 1]
 # Function f(x) = (x . b) mod 2 is guaranteed to be balanced
 b_int = random.randint(1, (1 << n_qubits) - 1)
 b_mask = format(b_int, f'0{n_qubits}b')
 # Add random bit flip on output (f(x) ^ c) where c in {0, 1}
 c_flip = random.choice([0, 1])
 if c_flip == 1:
  oracle_qc.x(n_qubits)
 for idx, bit in enumerate(b_mask):
  if bit == '1':
   oracle_qc.cx(idx, n_qubits)
 return oracle_qc, b_mask
n_test = 4
sim = AerSimulator()
print("--- Randomized Balanced Oracle Verification (n = 4) ---")
print(f"{'Trial':<8}{'Selected Bitmask b':<22}{'Measured String':<20}{'Result'}")
print("-" * 62)
for trial in range(1, 6):
 qc_rnd = QuantumCircuit(n_test + 1, n_test)
 qc_rnd.x(n_test)
 qc_rnd.h(range(n_test + 1))
 oracle_circ, mask = build_random_balanced_oracle(n_test)
 qc_rnd.compose(oracle_circ, inplace=True)
 qc_rnd.h(range(n_test))
 qc_rnd.measure(range(n_test), range(n_test))
 res = sim.run(qc_rnd, shots=500).result().get_counts()
 measured = list(res.keys())[0]
 verdict = "BALANCED (SUCCESS)" if measured != "0000" else "FAILED"
 print(f"{trial:<8}{mask:<22}{measured:<20}{verdict}")

--- Randomized Balanced Oracle Verification (n = 4) ---
Trial   Selected Bitmask b    Measured String     Result
--------------------------------------------------------------
1       1010                  0101                BALANCED (SUCCESS)
2       0001                  1000                BALANCED (SUCCESS)
3       0111                  1110                BALANCED (SUCCESS)
4       0001                  1000                BALANCED (SUCCESS)
5       0010                  0100                BALANCED (SUCCESS)
